# Predikcija sreće pojedinih zemalja
### Uvod u podatkovnu znanost — Tim 13

Skup podataka: **World Happiness Report (2015.–2019.)**

Ova bilježnica učitava podatke, čisti ih, prikazuje grafove te trenira i evaluira
regresijske (i jedan klasifikacijski) model. Pokreni ćelije redom (Shift + Enter).

> **Napomena o podacima:** CSV datoteke (`2015.csv` … `2019.csv`) moraju se nalaziti u
> podmapi `data/` pored ove bilježnice. U paketu koji si dobio one su već tu.


In [ ]:
# --- Biblioteke ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import collections

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              RandomForestClassifier)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta i izgled grafova
PRIMARY, SECOND, DARK, ACCENT, GRID = "#E8743B", "#F2C14E", "#2A3A4A", "#4C8C7D", "#D9DEE3"
plt.rcParams.update({
    "font.size": 11, "axes.edgecolor": DARK, "axes.labelcolor": DARK,
    "text.color": DARK, "xtick.color": DARK, "ytick.color": DARK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "figure.dpi": 110,
})

DATA = "data"   # podmapa s CSV datotekama

## 1. Učitavanje i objedinjavanje podataka
Nazivi stupaca razlikuju se kroz godine, pa ih ujednačavamo u jedinstvenu shemu i spajamo sve godine u jednu tablicu.

In [ ]:
COLS = ["country", "happiness_score", "gdp_per_capita", "social_support",
        "healthy_life_expectancy", "freedom", "generosity", "corruption", "year"]

def load_year(path, year):
    df = pd.read_csv(path)
    norm = {c: c.strip().lower().replace(".", " ").replace("(", " ").replace(")", " ") for c in df.columns}
    def find(keys):
        for c, n in norm.items():
            for k in keys:
                if k in n:
                    return c
        return None
    out = pd.DataFrame()
    out["country"] = df[find(["country"])]
    out["happiness_score"] = df[find(["happiness score", "score"])]
    out["gdp_per_capita"] = df[find(["economy", "gdp per capita"])]
    out["social_support"] = df[find(["family", "social support"])]
    out["healthy_life_expectancy"] = df[find(["health", "healthy life"])]
    out["freedom"] = df[find(["freedom"])]
    out["generosity"] = df[find(["generosity"])]
    out["corruption"] = df[find(["trust", "perceptions of corruption", "corruption"])]
    out["year"] = year
    return out[COLS]

frames = [load_year(f"{DATA}/{y}.csv", y) for y in [2015, 2016, 2017, 2018, 2019]]
raw = pd.concat(frames, ignore_index=True)

print("Objedinjeni (sirovi) skup:", raw.shape)
print("Broj zapisa po godini:")
print(raw["year"].value_counts().sort_index())
raw.head()

## 2. Čišćenje podataka
Provjeravamo duplikate, obrađujemo nedostajuće vrijednosti (medijan) te detektiramo i blago podrezujemo (winsorizacija) ekstremne vrijednosti metodom IQR.

In [ ]:
feature_cols = ["gdp_per_capita", "social_support", "healthy_life_expectancy",
                "freedom", "generosity", "corruption"]

# 2a) Duplikati
print("Potpuni duplikati redaka:", raw.duplicated().sum())
print("Duplikati po (zemlja, godina):", raw.duplicated(subset=["country", "year"]).sum())
df = raw.drop_duplicates().reset_index(drop=True)

# 2b) Nedostajuće vrijednosti
print("\nNedostajuće vrijednosti po koloni:")
print(df.isnull().sum()[df.isnull().sum() > 0])
for c in feature_cols + ["happiness_score"]:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())
print("Nedostajuće nakon obrade:", int(df.isnull().sum().sum()))

# 2c) Ekstremne vrijednosti (IQR) + winsorizacija značajki
print("\nBroj ekstremnih vrijednosti (IQR) po značajki:")
for c in feature_cols:
    q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[c] < low) | (df[c] > high)).sum())
    print(f"  {c}: {n_out}")
    df[c] = df[c].clip(lower=low, upper=high)   # winsorizacija

print("\nKonačan broj redaka:", df.shape[0])

## 3. Deskriptivna statistika

In [ ]:
df[["happiness_score"] + feature_cols].describe().T.round(3)

In [ ]:
corr = df[["happiness_score"] + feature_cols].corr()
print("Korelacija značajki s indeksom sreće (sortirano):")
corr["happiness_score"].drop("happiness_score").sort_values(ascending=False).round(3)

## 4. Vizualizacije

In [ ]:
LABELS = {
    "gdp_per_capita": "BDP po stanovniku", "social_support": "Socijalna podrška",
    "healthy_life_expectancy": "Očekivani zdravi život", "freedom": "Sloboda izbora",
    "generosity": "Velikodušnost", "corruption": "Percepcija korupcije",
    "happiness_score": "Indeks sreće",
}

# 4.1 Distribucija ciljne varijable
plt.figure(figsize=(7.2, 4.2))
plt.hist(df["happiness_score"], bins=25, color=PRIMARY, edgecolor="white", alpha=0.9)
plt.axvline(df["happiness_score"].mean(), color=DARK, ls="--", lw=1.6,
            label=f"Prosjek = {df['happiness_score'].mean():.2f}")
plt.xlabel("Indeks sreće"); plt.ylabel("Broj zemalja-godina")
plt.title("Distribucija indeksa sreće (2015.–2019.)", fontweight="bold")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 4.2 Korelacijska matrica
fig, ax = plt.subplots(figsize=(7.2, 6))
m = corr.values
im = ax.imshow(m, cmap="RdYlGn", vmin=-1, vmax=1)
labs = [LABELS[c] for c in corr.columns]
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs, fontsize=9)
for i in range(len(labs)):
    for j in range(len(labs)):
        ax.text(j, i, f"{m[i,j]:.2f}", ha="center", va="center",
                color="white" if abs(m[i,j]) > 0.6 else DARK, fontsize=8.5)
ax.set_title("Korelacijska matrica značajki", fontweight="bold", pad=12)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

In [ ]:
# 4.3 Raspršenja 4 najjače korelirane značajke vs cilj
top4 = corr["happiness_score"].drop("happiness_score").sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
for ax, c in zip(axes.ravel(), top4):
    ax.scatter(df[c], df["happiness_score"], s=18, color=ACCENT, alpha=0.55)
    z = np.polyfit(df[c], df["happiness_score"], 1)
    xs = np.linspace(df[c].min(), df[c].max(), 50)
    ax.plot(xs, np.polyval(z, xs), color=PRIMARY, lw=2)
    ax.set_xlabel(LABELS[c]); ax.set_ylabel("Indeks sreće")
    ax.set_title(f"{LABELS[c]} (r={corr['happiness_score'][c]:.2f})", fontsize=10.5)
fig.suptitle("Odnos ključnih značajki i indeksa sreće", fontweight="bold", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
# 4.4 Boxplotovi značajki
plt.figure(figsize=(8, 4.4))
bp = plt.boxplot([df[c] for c in feature_cols], patch_artist=True,
                 tick_labels=[LABELS[c] for c in feature_cols])
for p in bp["boxes"]: p.set_facecolor(SECOND); p.set_alpha(0.8)
for med in bp["medians"]: med.set_color(DARK); med.set_linewidth(1.6)
plt.xticks(rotation=25, ha="right", fontsize=9)
plt.title("Raspršenje značajki", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Regresija — treniranje i evaluacija
Dijelimo podatke 80:20, standardiziramo (za linearni model) te uspoređujemo tri modela uz petostruku unakrsnu validaciju.

In [ ]:
X = df[feature_cols].values
y = df["happiness_score"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Uzoraka za učenje:", len(y_train), "| za testiranje:", len(y_test))

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

models = {
    "Linearna regresija": (LinearRegression(), True),
    "Random Forest": (RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE), False),
    "Gradient Boosting": (GradientBoostingRegressor(random_state=RANDOM_STATE), False),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, preds = [], {}
for name, (mdl, scaled) in models.items():
    Xtr, Xte = (X_train_s, X_test_s) if scaled else (X_train, X_test)
    Xall = scaler.transform(X) if scaled else X
    mdl.fit(Xtr, y_train)
    p = mdl.predict(Xte); preds[name] = p
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, p),
        "MSE": mean_squared_error(y_test, p),
        "RMSE": np.sqrt(mean_squared_error(y_test, p)),
        "R2": r2_score(y_test, p),
        "CV_R2": cross_val_score(mdl, Xall, y, cv=cv, scoring="r2").mean(),
    })
results = pd.DataFrame(rows).set_index("Model").round(4)
results

In [ ]:
best = results["R2"].idxmax()
print("Najbolji model po R²:", best)

# Koeficijenti linearne regresije (standardizirano) i važnost značajki (RF)
lin = LinearRegression().fit(X_train_s, y_train)
coef = pd.Series(lin.coef_, index=[LABELS[c] for c in feature_cols]).sort_values(key=abs, ascending=False)
print("\nKoeficijenti linearne regresije (standardizirano):")
print(coef.round(3))

rf = models["Random Forest"][0]
imp = pd.Series(rf.feature_importances_, index=[LABELS[c] for c in feature_cols]).sort_values(ascending=False)
print("\nVažnost značajki (Random Forest):")
print(imp.round(3))

In [ ]:
# 5.1 Stvarno vs predviđeno (najbolji model)
p = preds[best]
plt.figure(figsize=(6.2, 6))
plt.scatter(y_test, p, s=26, color=PRIMARY, alpha=0.6, edgecolor="white")
lims = [min(y_test.min(), p.min()) - 0.2, max(y_test.max(), p.max()) + 0.2]
plt.plot(lims, lims, color=DARK, ls="--", lw=1.6, label="Idealna predikcija")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("Stvarni indeks sreće"); plt.ylabel("Predviđeni indeks sreće")
plt.title(f"Stvarne vs. predviđene vrijednosti\n({best}, R²={results.loc[best,'R2']:.3f})", fontweight="bold")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 5.2 Važnost značajki (RF)
plt.figure(figsize=(7.4, 4.2))
imp_sorted = imp.sort_values()
plt.barh(imp_sorted.index, imp_sorted.values, color=ACCENT, alpha=0.9, edgecolor="white")
for i, v in enumerate(imp_sorted.values):
    plt.text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=9)
plt.xlabel("Relativna važnost"); plt.xlim(0, imp.max() * 1.18)
plt.title("Važnost značajki (Random Forest)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# 5.3 Usporedba modela (R² i MAE)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.2, 4))
cols = [PRIMARY, ACCENT, SECOND]
a1.bar(results.index, results["R2"], color=cols, alpha=0.9, edgecolor="white")
a1.set_ylim(0, 1); a1.set_title("R² po modelu", fontweight="bold")
for i, v in enumerate(results["R2"]): a1.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
a1.tick_params(axis="x", rotation=12)
a2.bar(results.index, results["MAE"], color=cols, alpha=0.9, edgecolor="white")
a2.set_title("MAE po modelu (manje = bolje)", fontweight="bold")
for i, v in enumerate(results["MAE"]): a2.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
a2.tick_params(axis="x", rotation=12)
plt.tight_layout(); plt.show()

## 6. Klasifikacijska varijanta (poglavlje 4.3)
Indeks sreće diskretiziramo u tri razreda (niska / srednja / visoka) prema tercilima i treniramo klasifikator.

In [ ]:
q33, q66 = df["happiness_score"].quantile([1/3, 2/3]).values
def to_class(v):
    return "Niska" if v <= q33 else ("Srednja" if v <= q66 else "Visoka")
y_cls = df["happiness_score"].apply(to_class).values
print("Pragovi:", round(q33, 3), round(q66, 3))
print("Raspodjela razreda:", dict(collections.Counter(y_cls)))

Xtr, Xte, ytr, yte = train_test_split(X, y_cls, test_size=0.2,
                                      random_state=RANDOM_STATE, stratify=y_cls)
clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(Xtr, ytr)
pc = clf.predict(Xte)
order = ["Niska", "Srednja", "Visoka"]
print(f"\nTočnost:    {accuracy_score(yte, pc):.3f}")
print(f"Preciznost: {precision_score(yte, pc, average='macro', labels=order, zero_division=0):.3f}")
print(f"Odziv:      {recall_score(yte, pc, average='macro', labels=order, zero_division=0):.3f}")
print(f"F1-mjera:   {f1_score(yte, pc, average='macro', labels=order, zero_division=0):.3f}")

cm = confusion_matrix(yte, pc, labels=order)
plt.figure(figsize=(5.6, 5))
plt.imshow(cm, cmap="Oranges")
plt.xticks(range(3), order); plt.yticks(range(3), order)
plt.xlabel("Predviđeni razred"); plt.ylabel("Stvarni razred")
plt.title(f"Matrica zabune (točnost = {accuracy_score(yte, pc):.1%})", fontweight="bold")
thr = cm.max() / 2
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > thr else DARK, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Sažetak

- Najbolji regresijski model je **Random Forest** (najviši R², najniži MAE).
- Najutjecajniji čimbenici sreće: **BDP po stanovniku** i **očekivani zdravi život**.
- Klasifikacijska varijanta (3 razreda) postiže visoku točnost, uz pogreške gotovo isključivo između susjednih razreda.

> Brojevi koje ovdje dobiješ identični su onima u projektnoj dokumentaciji (`random_state = 42`).
